In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df=pd.read_csv("/content/diabetic_patient_data.csv")

In [3]:
df.sample(5)

,encounter_id,patient_nbr,race,gender,age,weight,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_desc,discharge_disposition_desc,admission_source_desc
49815,150390360,24861600,Caucasian,Female,[80-90),?,5,MC,InternalMedicine,56,...,No,No,No,No,Ch,Yes,NO,Emergency,Discharged/transferred to a long term care hos...,Emergency Room
20498,72186084,5845482,Hispanic,Female,[60-70),?,3,?,?,34,...,No,No,No,No,No,No,NO,Emergency,Discharged to home,Emergency Room
49183,149160474,106442883,Caucasian,Female,[70-80),?,7,MC,Emergency/Trauma,47,...,No,No,No,No,Ch,Yes,NO,Elective,Discharged/transferred to SNF,Physician Referral
37651,116911020,23233374,AfricanAmerican,Male,[80-90),?,5,MC,Family/GeneralPractice,43,...,No,No,No,No,No,No,NO,Emergency,Discharged/transferred to another rehab fac in...,Emergency Room
9835,42258804,3879450,Caucasian,Female,[70-80),?,9,?,Family/GeneralPractice,64,...,No,No,No,No,Ch,Yes,>30,Emergency,NaN,Emergency Room


In [4]:
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
time_in_hospital,0
payer_code,0
medical_specialty,0
num_lab_procedures,0


In [5]:
df['admission_type_desc'].fillna(df['admission_type_desc'].mode()[0],inplace=True)
df['discharge_disposition_desc'].fillna(df['discharge_disposition_desc'].mode()[0],inplace=True)

/tmp/ipykernel_470/3224064984.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['admission_type_desc'].fillna(df['admission_type_desc'].mode()[0],inplace=True)
/tmp/ipykernel_470/3224064984.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].me

In [7]:
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
time_in_hospital,0
payer_code,0
medical_specialty,0
num_lab_procedures,0


In [8]:
# Drop columns that are identifiers or have very little predictive value
drop_cols = [
    'encounter_id',      # Unique encounter ID, not a clinical feature
    'patient_nbr',       # Patient ID, can cause data leakage
    'weight',            # Extremely high missingness in the original dataset
    'payer_code',        # Administrative information, not very useful for first model
    'medical_specialty', # Very high missingness / many categories
    'examide',           # Essentially no useful variation
    'citoglipton'        # Essentially no useful variation
]

df = df.drop(columns=drop_cols)

In [9]:
# Columns with only a few missing values
mode_cols = [
    'glipizide-metformin',
    'glimepiride-pioglitazone',
    'metformin-rosiglitazone',
    'metformin-pioglitazone',
    'change',
    'diabetesMed'
]

for col in mode_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
df['max_glu_serum'] = df['max_glu_serum'].fillna('Not Tested')
df['A1Cresult'] = df['A1Cresult'].fillna('Not Tested')

In [11]:
desc_cols = [
    'admission_type_desc',
    'discharge_disposition_desc',
    'admission_source_desc'
]

for col in desc_cols:
    df[col] = df[col].fillna('Unknown')

In [12]:
df = df.dropna(subset=['readmitted'])

In [13]:
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

# Drop original target
df = df.drop(columns=['readmitted'])

In [16]:
df.isnull().sum()

,0
race,0
gender,0
age,0
time_in_hospital,0
num_lab_procedures,0
num_procedures,0
num_medications,0
number_outpatient,0
number_emergency,0
number_inpatient,0


In [18]:
df.columns

Index(['race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin',
       'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
       'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed',
       'admission_type_desc', 'discharge_disposition_desc',
       'admission_source_desc', 'readmitted_30'],
      dtype='object')

In [22]:
X = df.drop(columns=['readmitted_30'])
y = df['readmitted_30']

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(exclude=['object']).columns

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)

Categorical columns:
Index(['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
       'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed',
       'admission_type_desc', 'discharge_disposition_desc',
       'admission_source_desc'],
      dtype='object')

Numerical columns:
Index(['time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses'],
      dtype='object')


In [26]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore')

In [27]:
df

,race,gender,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,admission_type_desc,discharge_disposition_desc,admission_source_desc,readmitted_30
0,Caucasian,Female,[0-10),1,41,0,1,0,0,0,...,No,No,No,No,No,No,Emergency,Not Mapped,Physician Referral,0
1,Caucasian,Female,[10-20),3,59,0,18,0,0,0,...,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room,0
2,AfricanAmerican,Female,[20-30),2,11,5,13,2,0,1,...,No,No,No,No,No,Yes,Emergency,Discharged to home,Emergency Room,0
3,Caucasian,Male,[30-40),2,44,1,16,0,0,0,...,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room,0
4,Caucasian,Male,[40-50),1,51,0,8,0,0,0,...,No,No,No,No,Ch,Yes,Emergency,Discharged to home,Emergency Room,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82787,AfricanAmerican,Male,[40-50),2,39,1,6,1,1,1,...,No,No,No,No,No,No,Elective,Discharged/transferred to home with home healt...,Physician Referral,0
82788,Caucasian,Female,[80-90),13,51,1,14,2,0,0,...,No,No,No,No,Ch,Yes,Elective,Discharged/transferred to SNF,Physician Referral,0
82789,Caucasian,Female,[80-90),2,39,1,15,0,0,1,...,No,No,No,No,No,Yes,Urgent,Discharged to home,Emergency Room,0
82790,AfricanAmerican,Female,[70-80),6,1,2,12,0,0,0,...,No,No,No,No,No,Yes,Urgent,Discharged/transferred to SNF,Physician Referral,1


In [28]:
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

In [29]:
print(X_train_cat.shape)
print(X_test_cat.shape)

(66233, 2232)
(16559, 2232)


In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[numerical_cols])
X_test_num = scaler.transform(X_test[numerical_cols])

In [31]:
print(X_train_num.shape)
print(X_test_num.shape)

(66233, 8)
(16559, 8)


In [32]:
from scipy.sparse import hstack

X_train_processed = hstack([X_train_num, X_train_cat])
X_test_processed = hstack([X_test_num, X_test_cat])

In [33]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    penalty='l2',
    C=1.0,
    max_iter=1000
)

In [34]:
model.fit(X_train_processed, y_train)

LogisticRegression(max_iter=1000)

In [35]:
y_prob = model.predict_proba(X_test_processed)[:, 1]

In [36]:
y_prob

array([0.22783127, 0.15189918, 0.16544742, ..., 0.08961682, 0.13686739,
       0.17523712])

In [37]:
print(y_prob[:10])

[0.22783127 0.15189918 0.16544742 0.03715926 0.05691381 0.13331324
 0.39203587 0.29656012 0.10850474 0.03991866]


In [38]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.6614039364721438


In [39]:
y_pred = model.predict(X_test_processed)

In [40]:
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [41]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[14651    42]
 [ 1818    48]]


In [42]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8876743764720092
